# 07. TTL-Aware SCARF Defense

This notebook implements the novel TTL-aware SCARF architecture combining contrastive representation learning, perturbation loss, and consistency regularization. It conducts multi-seed training, Difference-in-Differences statistical testing, and summarizes clean vs perturbed performance.


In [47]:
# =============================================================================
# FINAL SCARF COMPARISON — replaces cells 45 through 56 entirely.
# Trains and evaluates BOTH models explicitly, no globals() guessing.
# Requires from earlier cells: X_train_s, X_test_s, y_train_s, y_test_s,
# train_df, preprocessor, SCARFEncoderMulti, SCARFClassifierMulti,
# HIDDEN_DIM, EMBED_DIM, device_scarf_multi, dttl_test_sets
# =============================================================================

import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

SEEDS = [42, 123, 2026]
DTTL_LEVELS = [-2, -5, -10, -15]
BATCH_SIZE = 512
LR = 1e-4
CONTRASTIVE_EPOCHS = 10
FINETUNE_EPOCHS = 10
CORRUPTION_RATE = 0.6
TEMPERATURE = 0.5
PERT_WEIGHT = 0.40
CONS_WEIGHT = 0.15

device = device_scarf_multi


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def scarf_corrupt(x, rate):
    b, f = x.shape
    mask = torch.rand(b, f, device=x.device) < rate
    idx = torch.randint(0, b, (b, f), device=x.device)
    return torch.where(mask, x[idx, torch.arange(f, device=x.device)], x)


def nt_xent(z1, z2, temp):
    z1, z2 = F.normalize(z1, dim=1), F.normalize(z2, dim=1)
    b = z1.shape[0]
    reps = torch.cat([z1, z2], dim=0)
    sim = torch.matmul(reps, reps.T) / temp
    sim.fill_diagonal_(-1e9)
    targets = torch.arange(b, device=z1.device)
    targets = torch.cat([targets + b, targets], dim=0)
    return F.cross_entropy(sim, targets)


def evaluate(model, X, y):
    model.eval()
    X_t = torch.tensor(np.asarray(X, dtype=np.float32), device=device)
    y_np = np.asarray(y).reshape(-1).astype(int)
    with torch.no_grad():
        probs = torch.sigmoid(model(X_t).reshape(-1)).cpu().numpy()
    preds = (probs >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_np, preds, labels=[0, 1]).ravel()
    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    auc = roc_auc_score(y_np, probs)
    ap = average_precision_score(y_np, probs)
    return tpr, fpr, auc, ap


def make_model():
    encoder = SCARFEncoderMulti(input_dim=X_train_s.shape[1],
                                 hidden_dim=HIDDEN_DIM, embed_dim=EMBED_DIM).to(device)
    return SCARFClassifierMulti(encoder, EMBED_DIM).to(device)


def get_pos_weight():
    pos = float(np.sum(np.asarray(y_train_s)))
    neg = float(len(y_train_s) - pos)
    return torch.tensor([neg / pos], dtype=torch.float32, device=device)


def build_ttl_perturbed_training_sets():
    """Raw-feature dTTL perturbation of the TRAINING set, matching
    Section 3.3.3's protocol, reused for TTL-aware training."""
    dttl_col = [c for c in train_df.columns if "dttl" in str(c).lower()][0]
    target_col = "label" if "label" in train_df.columns else "Label"
    out = {}
    for pct in DTTL_LEVELS:
        df = train_df.copy()
        df[dttl_col] = pd.to_numeric(df[dttl_col], errors="coerce") * (1 + pct / 100.0)
        df[dttl_col] = df[dttl_col].round().clip(lower=1, upper=255)
        X_raw = df.drop(columns=[target_col])
        processed = preprocessor.transform(X_raw)
        if hasattr(processed, "toarray"):
            processed = processed.toarray()
        out[pct] = np.asarray(processed, dtype=np.float32)
    return out


# =============================================================================
# MODEL A — ORIGINAL SCARF (contrastive pretrain + finetune, no TTL exposure)
# =============================================================================

def train_original_scarf(seed):
    print(f"\n[Original SCARF] seed {seed}")
    set_seed(seed)
    model = make_model()

    X_t = torch.tensor(np.asarray(X_train_s, dtype=np.float32))
    y_t = torch.tensor(np.asarray(y_train_s, dtype=np.float32)).reshape(-1)
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=BATCH_SIZE,
                         shuffle=True, drop_last=True)

    pretrain_opt = torch.optim.Adam(model.encoder.parameters(), lr=LR)
    for epoch in range(CONTRASTIVE_EPOCHS):
        model.encoder.train()
        for xb, _ in loader:
            xb = xb.to(device)
            z1 = model.encoder(scarf_corrupt(xb, CORRUPTION_RATE))
            z2 = model.encoder(scarf_corrupt(xb, CORRUPTION_RATE))
            loss = nt_xent(z1, z2, TEMPERATURE)
            pretrain_opt.zero_grad(); loss.backward(); pretrain_opt.step()

    criterion = nn.BCEWithLogitsLoss(pos_weight=get_pos_weight())
    finetune_opt = torch.optim.Adam(model.parameters(), lr=LR)
    for epoch in range(FINETUNE_EPOCHS):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = criterion(model(xb).reshape(-1), yb)
            finetune_opt.zero_grad(); loss.backward(); finetune_opt.step()

    return model


# =============================================================================
# MODEL B — TTL-AWARE SCARF (same contrastive stage + TTL-perturbed finetune)
# =============================================================================
# =============================================================================
# CORRECTED TTL-AWARE SCARF TRAINING
# =============================================================================

def train_ttl_aware_scarf(seed, pert_tensors):
    print(f"\n[TTL-aware SCARF] seed {seed}")

    set_seed(seed)

    model = make_model()

    # ---------------------------------------------------------
    # Clean training data WITH SAMPLE INDICES
    # ---------------------------------------------------------

    X_t = torch.tensor(
        np.asarray(X_train_s, dtype=np.float32)
    )

    y_t = torch.tensor(
        np.asarray(y_train_s, dtype=np.float32)
    ).reshape(-1)

    index_t = torch.arange(len(X_t))

    dataset = TensorDataset(
        X_t,
        y_t,
        index_t
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=True
    )

    # ---------------------------------------------------------
    # CONTRASTIVE PRETRAINING
    # Same as Original SCARF
    # ---------------------------------------------------------

    pretrain_opt = torch.optim.Adam(
        model.encoder.parameters(),
        lr=LR
    )

    for epoch in range(CONTRASTIVE_EPOCHS):

        model.encoder.train()

        for xb, _, _ in loader:

            xb = xb.to(device)

            z1 = model.encoder(
                scarf_corrupt(xb, CORRUPTION_RATE)
            )

            z2 = model.encoder(
                scarf_corrupt(xb, CORRUPTION_RATE)
            )

            loss = nt_xent(
                z1,
                z2,
                TEMPERATURE
            )

            pretrain_opt.zero_grad()
            loss.backward()
            pretrain_opt.step()

    # ---------------------------------------------------------
    # SUPERVISED TTL-AWARE FINE-TUNING
    # ---------------------------------------------------------

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=get_pos_weight()
    )

    finetune_opt = torch.optim.Adam(
        model.parameters(),
        lr=LR
    )

    for epoch in range(FINETUNE_EPOCHS):

        model.train()

        for xb, yb, batch_indices in loader:

            xb = xb.to(device)
            yb = yb.to(device)

            # -------------------------------------------------
            # CLEAN PREDICTION
            # -------------------------------------------------

            logits_clean = model(xb).reshape(-1)

            clean_loss = criterion(
                logits_clean,
                yb
            )

            # -------------------------------------------------
            # TTL PERTURBATION LOSS
            #
            # IMPORTANT:
            # SAME SAMPLE INDICES ARE USED.
            # Therefore labels remain correctly aligned.
            # -------------------------------------------------

            pert_losses = []
            pert_probs = []

            for level in DTTL_LEVELS:

                pert_pool = pert_tensors[level]

                # Same samples, same ordering
                xb_pert = pert_pool[batch_indices].to(device)

                logits_pert = model(
                    xb_pert
                ).reshape(-1)

                pert_loss = criterion(
                    logits_pert,
                    yb
                )

                pert_losses.append(
                    pert_loss
                )

                pert_probs.append(
                    torch.sigmoid(logits_pert)
                )

            # Mean loss across all four dTTL conditions
            pert_loss = torch.stack(
                pert_losses
            ).mean()

            # -------------------------------------------------
            # CONSISTENCY LOSS
            # Force predictions to remain stable under
            # controlled dTTL perturbations.
            # -------------------------------------------------

            clean_prob = torch.sigmoid(
                logits_clean
            )

            consistency_losses = []

            for pert_prob in pert_probs:

                consistency_losses.append(
                    torch.mean(
                        (clean_prob - pert_prob) ** 2
                    )
                )

            cons_loss = torch.stack(
                consistency_losses
            ).mean()

            # -------------------------------------------------
            # FINAL TTL-AWARE OBJECTIVE
            # -------------------------------------------------

            loss = (
                clean_loss
                + PERT_WEIGHT * pert_loss
                + CONS_WEIGHT * cons_loss
            )

            finetune_opt.zero_grad()
            loss.backward()
            finetune_opt.step()

    return model

# =============================================================================
# RUN
# =============================================================================

print("Building TTL-perturbed training sets...")
ttl_train_np = build_ttl_perturbed_training_sets()
ttl_train_tensors = {k: torch.tensor(v, dtype=torch.float32) for k, v in ttl_train_np.items()}

rows = []
for seed in SEEDS:
    orig_model = train_original_scarf(seed)
    ttl_model = train_ttl_aware_scarf(seed, ttl_train_tensors)

    orig_clean = evaluate(orig_model, X_test_s, y_test_s)
    ttl_clean = evaluate(ttl_model, X_test_s, y_test_s)

    rows.append({"Seed": seed, "Model": "Original", "dTTL_%": 0,
                 "TPR": orig_clean[0], "FPR": orig_clean[1],
                 "AUROC": orig_clean[2], "AUPRC": orig_clean[3],
                 "TPR_degradation": 0.0})
    rows.append({"Seed": seed, "Model": "TTL-aware", "dTTL_%": 0,
                 "TPR": ttl_clean[0], "FPR": ttl_clean[1],
                 "AUROC": ttl_clean[2], "AUPRC": ttl_clean[3],
                 "TPR_degradation": 0.0})

    for level in DTTL_LEVELS:
        X_pert = dttl_test_sets[level]

        o = evaluate(orig_model, X_pert, y_test_s)
        rows.append({"Seed": seed, "Model": "Original", "dTTL_%": level,
                     "TPR": o[0], "FPR": o[1], "AUROC": o[2], "AUPRC": o[3],
                     "TPR_degradation": orig_clean[0] - o[0]})

        t = evaluate(ttl_model, X_pert, y_test_s)
        rows.append({"Seed": seed, "Model": "TTL-aware", "dTTL_%": level,
                     "TPR": t[0], "FPR": t[1], "AUROC": t[2], "AUPRC": t[3],
                     "TPR_degradation": ttl_clean[0] - t[0]})

final_df = pd.DataFrame(rows)
final_df.to_csv("FINAL_scarf_comparison_raw.csv", index=False)

summary = (final_df.groupby(["Model", "dTTL_%"])
           [["TPR", "AUROC", "AUPRC", "TPR_degradation"]]
           .agg(["mean", "std"]))
summary.to_csv("FINAL_scarf_comparison_summary.csv")

print("\n" + "=" * 80)
print("FINAL SUMMARY (mean ± std across seeds 42, 123, 2026)")
print("=" * 80)
print(summary.round(4).to_string())

Building TTL-perturbed training sets...

[Original SCARF] seed 42

[TTL-aware SCARF] seed 42

[Original SCARF] seed 123

[TTL-aware SCARF] seed 123

[Original SCARF] seed 2026

[TTL-aware SCARF] seed 2026

FINAL SUMMARY (mean ± std across seeds 42, 123, 2026)
                     TPR           AUROC           AUPRC         TPR_degradation        
                    mean     std    mean     std    mean     std            mean     std
Model     dTTL_%                                                                        
Original  -15     0.7084  0.0272  0.8910  0.0091  0.9166  0.0084          0.0502  0.0020
          -10     0.7252  0.0281  0.8918  0.0095  0.9169  0.0089          0.0333  0.0017
          -5      0.7411  0.0281  0.8924  0.0101  0.9171  0.0094          0.0174  0.0018
          -2      0.7518  0.0284  0.8927  0.0105  0.9171  0.0098          0.0068  0.0007
           0      0.7585  0.0291  0.8927  0.0107  0.9171  0.0101          0.0000  0.0000
TTL-aware -15     0.7277  0.

In [41]:
# ================================================================
# M7.1 — CORRECT STATISTICAL TEST OF ROBUSTNESS IMPROVEMENT
#
# Difference-in-differences:
#
#   (SCARF perturbed - SCARF clean)
# - (Standard perturbed - Standard clean)
#
# Positive value = SCARF degraded LESS than Standard.
# ================================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("M7.1 — DIFFERENCE-IN-DIFFERENCES ROBUSTNESS TEST")
print("=" * 80)

N_BOOT = 2000
rng = np.random.default_rng(2026)

# ------------------------------------------------
# TPR CHANGE FOR EACH INDIVIDUAL SAMPLE
# ------------------------------------------------

def binary_tpr(y_true, probs, threshold=0.5):

    pred = (probs >= threshold).astype(int)

    positive = (y_true == 1)

    if positive.sum() == 0:
        return np.nan

    return (
        pred[positive] == 1
    ).mean()


# ------------------------------------------------
# BOOTSTRAP ROBUSTNESS EFFECT
# ------------------------------------------------

rows = []

negative_levels = [-2, -5, -10, -15]

for pct in negative_levels:

    print(f"\nTesting dTTL {pct:+d}% ...")

    std_clean = prediction_store[0]["standard"]
    std_pert  = prediction_store[pct]["standard"]

    scarf_clean = prediction_store[0]["scarf"]
    scarf_pert  = prediction_store[pct]["scarf"]

    # ------------------------------------------------
    # Observed robustness changes
    # ------------------------------------------------

    std_clean_tpr = binary_tpr(
        y,
        std_clean
    )

    std_pert_tpr = binary_tpr(
        y,
        std_pert
    )

    scarf_clean_tpr = binary_tpr(
        y,
        scarf_clean
    )

    scarf_pert_tpr = binary_tpr(
        y,
        scarf_pert
    )

    std_change = (
        std_pert_tpr -
        std_clean_tpr
    )

    scarf_change = (
        scarf_pert_tpr -
        scarf_clean_tpr
    )

    observed_effect = (
        scarf_change -
        std_change
    )

    # ------------------------------------------------
    # PAIRED BOOTSTRAP
    # ------------------------------------------------

    n = len(y)

    bootstrap_effects = []

    for _ in range(N_BOOT):

        idx = rng.integers(
            0,
            n,
            size=n
        )

        y_b = y[idx]

        try:

            std_clean_b = binary_tpr(
                y_b,
                std_clean[idx]
            )

            std_pert_b = binary_tpr(
                y_b,
                std_pert[idx]
            )

            scarf_clean_b = binary_tpr(
                y_b,
                scarf_clean[idx]
            )

            scarf_pert_b = binary_tpr(
                y_b,
                scarf_pert[idx]
            )

            std_change_b = (
                std_pert_b -
                std_clean_b
            )

            scarf_change_b = (
                scarf_pert_b -
                scarf_clean_b
            )

            effect_b = (
                scarf_change_b -
                std_change_b
            )

            bootstrap_effects.append(
                effect_b
            )

        except Exception:
            continue

    bootstrap_effects = np.asarray(
        bootstrap_effects
    )

    ci_low = np.percentile(
        bootstrap_effects,
        2.5
    )

    ci_high = np.percentile(
        bootstrap_effects,
        97.5
    )

    significant = (
        ci_low > 0
        or
        ci_high < 0
    )

    # ------------------------------------------------
    # RELATIVE ROBUSTNESS IMPROVEMENT
    # ------------------------------------------------

    std_loss = abs(std_change)

    if std_loss > 0:

        relative_improvement = (
            observed_effect /
            std_loss
        ) * 100

    else:

        relative_improvement = np.nan

    rows.append({

        "dTTL_%": pct,

        "Standard_clean_TPR":
            std_clean_tpr,

        "Standard_perturbed_TPR":
            std_pert_tpr,

        "Standard_TPR_change":
            std_change,

        "SCARF_clean_TPR":
            scarf_clean_tpr,

        "SCARF_perturbed_TPR":
            scarf_pert_tpr,

        "SCARF_TPR_change":
            scarf_change,

        "Robustness_effect_SCARF_minus_Standard":
            observed_effect,

        "CI_95_low":
            ci_low,

        "CI_95_high":
            ci_high,

        "Relative_robustness_improvement_%":
            relative_improvement,

        "Significant_95pct":
            significant
    })


m71_df = pd.DataFrame(rows)


# ================================================================
# DISPLAY
# ================================================================

print("\n")
print("=" * 80)
print("CORRECT ROBUSTNESS SIGNIFICANCE RESULTS")
print("=" * 80)

print(
    m71_df.round(6).to_string(
        index=False
    )
)


# ================================================================
# SIMPLE INTERPRETATION
# ================================================================

print("\n")
print("=" * 80)
print("INTERPRETATION")
print("=" * 80)

for _, row in m71_df.iterrows():

    pct = int(row["dTTL_%"])

    effect = row[
        "Robustness_effect_SCARF_minus_Standard"
    ]

    low = row["CI_95_low"]
    high = row["CI_95_high"]

    improvement = row[
        "Relative_robustness_improvement_%"
    ]

    sig = row[
        "Significant_95pct"
    ]

    print(
        f"dTTL {pct:+d}% | "
        f"robustness effect = {effect:+.6f} | "
        f"95% CI = [{low:+.6f}, {high:+.6f}] | "
        f"relative improvement = {improvement:.2f}% | "
        f"significant = {sig}"
    )


# ================================================================
# SAVE
# ================================================================

m71_df.to_csv(
    "M7_1_difference_in_differences_robustness.csv",
    index=False
)

print("\nSaved:")
print(
    "M7_1_difference_in_differences_robustness.csv"
)

print("\n" + "=" * 80)
print("M7.1 COMPLETE")
print("=" * 80)

M7.1 — DIFFERENCE-IN-DIFFERENCES ROBUSTNESS TEST

Testing dTTL -2% ...

Testing dTTL -5% ...

Testing dTTL -10% ...

Testing dTTL -15% ...


CORRECT ROBUSTNESS SIGNIFICANCE RESULTS
 dTTL_%  Standard_clean_TPR  Standard_perturbed_TPR  Standard_TPR_change  SCARF_clean_TPR  SCARF_perturbed_TPR  SCARF_TPR_change  Robustness_effect_SCARF_minus_Standard  CI_95_low  CI_95_high  Relative_robustness_improvement_%  Significant_95pct
     -2            0.429167                0.423762            -0.005405         0.461969             0.456366         -0.005603                               -0.000199  -0.001143    0.000684                          -3.673469              False
     -5            0.429167                0.415005            -0.014162         0.461969             0.447653         -0.014317                               -0.000154  -0.001503    0.001256                          -1.090343              False
    -10            0.429167                0.403225            -0.025942         

In [49]:
# =============================================================================
# 3-SEED TTL-AWARE SCARF SUMMARY TABLE
# =============================================================================

print("\n" + "=" * 90)
print("TTL-AWARE SCARF — THREE-SEED CLEAN TEST SUMMARY")
print("=" * 90)

# -------------------------------------------------------------------------
# 1. Extract clean-test performance for TTL-aware SCARF for each seed
# -------------------------------------------------------------------------

ttl_clean_summary = (
    final_df[
        (final_df["Model"] == "TTL-aware") &
        (final_df["dTTL_%"] == 0)
    ][
        ["Seed", "TPR", "FPR", "AUROC", "AUPRC"]
    ]
    .sort_values("Seed")
    .reset_index(drop=True)
)

print("\nPer-seed clean-test performance:")
print(ttl_clean_summary.round(4).to_string(index=False))

ttl_clean_summary.to_csv(
    "FINAL_TTL_aware_SCARF_three_seed_clean_summary.csv",
    index=False
)


# =============================================================================
# 2. THREE-SEED ROBUSTNESS SUMMARY
# =============================================================================
# For each seed, calculate the mean TPR degradation over the four evaluated
# negative dTTL perturbations: -2%, -5%, -10%, -15%.
#
# Lower TPR degradation = better robustness.
# -------------------------------------------------------------------------

negative_levels = [-2, -5, -10, -15]

ttl_robustness_summary = (
    final_df[
        (final_df["Model"] == "TTL-aware") &
        (final_df["dTTL_%"].isin(negative_levels))
    ]
    .groupby("Seed")
    .agg(
        Clean_TPR=("TPR", lambda x: np.nan),  # replaced below
        Mean_TPR=("TPR", "mean"),
        Mean_TPR_degradation=("TPR_degradation", "mean"),
        Worst_TPR_degradation=("TPR_degradation", "max"),
        Mean_AUROC=("AUROC", "mean"),
        Mean_AUPRC=("AUPRC", "mean")
    )
    .reset_index()
)

# Add the actual clean TPR for each seed
clean_tpr_by_seed = (
    final_df[
        (final_df["Model"] == "TTL-aware") &
        (final_df["dTTL_%"] == 0)
    ][["Seed", "TPR"]]
    .rename(columns={"TPR": "Clean_TPR"})
)

ttl_robustness_summary = ttl_robustness_summary.drop(
    columns=["Clean_TPR"]
).merge(
    clean_tpr_by_seed,
    on="Seed",
    how="left"
)

# Reorder columns
ttl_robustness_summary = ttl_robustness_summary[
    [
        "Seed",
        "Clean_TPR",
        "Mean_TPR",
        "Mean_TPR_degradation",
        "Worst_TPR_degradation",
        "Mean_AUROC",
        "Mean_AUPRC"
    ]
].sort_values("Seed")

print("\nThree-seed robustness summary:")
print(ttl_robustness_summary.round(4).to_string(index=False))

ttl_robustness_summary.to_csv(
    "FINAL_TTL_aware_SCARF_three_seed_robustness_summary.csv",
    index=False
)


# =============================================================================
# 3. OVERALL THREE-SEED MEAN ± SD
# =============================================================================

ttl_overall = (
    ttl_clean_summary[
        ["TPR", "FPR", "AUROC", "AUPRC"]
    ]
    .agg(["mean", "std"])
)

print("\n" + "=" * 90)
print("TTL-AWARE SCARF — OVERALL THREE-SEED MEAN ± SD")
print("=" * 90)
print(ttl_overall.round(4).to_string())


# =============================================================================
# 4. SAVE FINAL PAPER-READY SUMMARY
# =============================================================================

paper_ttl_summary = pd.DataFrame({
    "Metric": ["TPR", "FPR", "AUROC", "AUPRC"],
    "Mean": [
        ttl_clean_summary["TPR"].mean(),
        ttl_clean_summary["FPR"].mean(),
        ttl_clean_summary["AUROC"].mean(),
        ttl_clean_summary["AUPRC"].mean()
    ],
    "SD": [
        ttl_clean_summary["TPR"].std(),
        ttl_clean_summary["FPR"].std(),
        ttl_clean_summary["AUROC"].std(),
        ttl_clean_summary["AUPRC"].std()
    ]
})

paper_ttl_summary.to_csv(
    "FINAL_TTL_aware_SCARF_paper_summary.csv",
    index=False
)

print("\nSaved:")
print("FINAL_TTL_aware_SCARF_three_seed_clean_summary.csv")
print("FINAL_TTL_aware_SCARF_three_seed_robustness_summary.csv")
print("FINAL_TTL_aware_SCARF_paper_summary.csv")


TTL-AWARE SCARF — THREE-SEED CLEAN TEST SUMMARY

Per-seed clean-test performance:
 Seed    TPR    FPR  AUROC  AUPRC
   42 0.7648 0.2493 0.8860 0.9093
  123 0.7874 0.2429 0.9027 0.9262
 2026 0.7265 0.2280 0.8842 0.9108

Three-seed robustness summary:
 Seed  Clean_TPR  Mean_TPR  Mean_TPR_degradation  Worst_TPR_degradation  Mean_AUROC  Mean_AUPRC
   42     0.7648    0.7495                0.0153                 0.0292      0.8857      0.9095
  123     0.7874    0.7683                0.0191                 0.0349      0.9013      0.9252
 2026     0.7265    0.7099                0.0165                 0.0315      0.8839      0.9108

TTL-AWARE SCARF — OVERALL THREE-SEED MEAN ± SD
         TPR     FPR   AUROC   AUPRC
mean  0.7596  0.2401  0.8910  0.9154
std   0.0308  0.0110  0.0102  0.0094

Saved:
FINAL_TTL_aware_SCARF_three_seed_clean_summary.csv
FINAL_TTL_aware_SCARF_three_seed_robustness_summary.csv
FINAL_TTL_aware_SCARF_paper_summary.csv
